In [0]:
dbutils.fs.ls('abfss://aqi-data@lshc.dfs.core.windows.net/cpcb_raw/year=2026/month=09/')

In [0]:
raw_path = "abfss://aqi-data@lshc.dfs.core.windows.net/cpcb_raw/*/*/*/*.json"

# Enable multiline parsing so Spark reads the entire indented JSON document as one object
raw_df = (
    spark.read
    .option("multiline", "true")
    .format("json")
    .load(raw_path)
)

raw_df.printSchema()
display(raw_df)

In [0]:
from pyspark.sql.functions import col, explode, current_timestamp

flattened_df = (
    raw_df
    .withColumn("record", explode(col("records")))
    .select(
        col("record.country").alias("country"),
        col("record.state").alias("state"),
        col("record.city").alias("city"),
        col("record.station").alias("station"),
        col("record.latitude").try_cast("double").alias("latitude"),
        col("record.longitude").try_cast("double").alias("longitude"),
        col("record.last_update").alias("last_update"),
        col("record.pollutant_id").alias("pollutant_id"),
        col("record.min_value").try_cast("double").alias("pollutant_min"),
        col("record.max_value").try_cast("double").alias("pollutant_max"),
        col("record.avg_value").try_cast("double").alias("pollutant_avg"),
        current_timestamp().alias("ingested_at"),
        col("_metadata.file_path").alias("source_file")
    )
)

display(flattened_df)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS lshc_lakehouse.cpcb_aqi_lake
MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables';